# TypeIa SN models

### The two TypeIa supernova models described in the papers Warren P. Wright et al., Phys. Rev. D94 (2016) 025026, and Warren P. Wright et al.,Phys. Rev. D95 (2017) 043006

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('font', size=14)

import numpy as np

from astropy import units as u 

from snewpy.models.typeIa import Wright_2016
from snewpy.flavor import ThreeFlavor
from snewpy.neutrino import MixingParameters
from snewpy.flavor_transformation import AdiabaticMSW
from snewpy.rate_calculator import RateCalculator, aggregate

## A Function which plots the Initial and Oscillated Spectra using the Adiabatic MSW Flavor Transformation and the Normal mass ordering

Plot the neutrino spectra at the source and after the requested flavor transformation has been applied.

In [ ]:
def plot_spectra(model, time=1*u.s):
    # Adiabatic MSW effect. NMO is used by default.
    xform_nmo = AdiabaticMSW(MixingParameters())

    # Energy array and time to compute spectra.
    # Note that any convenient units can be used and the calculation will remain internally consistent.
    E = np.linspace(0,10,201) * u.MeV

    ispec = model.get_initial_spectra(time, E)
    ospec_nmo = model.get_transformed_spectra(time, E, xform_nmo)

    fig, axes = plt.subplots(1,2, figsize=(12,5), sharex=True, sharey=True, tight_layout=True)

    for i, spec in enumerate([ispec, ospec_nmo]):
        ax = axes[i]
        for flavor in ThreeFlavor:
            ax.plot(E, spec[flavor].array.squeeze(),
                   label=flavor.to_tex(),
                   color='C0' if flavor.is_electron else 'C1',
                   ls='-' if flavor.is_neutrino else ':', lw=2,
                   alpha=0.7)

        ax.set(xlabel=r'$E$ [{}]'.format(E.unit),
           title='Initial Spectra: $t = ${:.1f}'.format(time) if i==0 else 'Oscillated Spectra: $t = ${:.1f}'.format(time))
        ax.grid()
        ax.legend(loc='upper right', ncol=2, fontsize=16)

    ax = axes[0]
    ax.set(ylabel=r'neutrino spectra [erg$^{-1}$ s$^{-1}$]')

    fig.tight_layout();

## Look at what models are available

In [ ]:
Wright_2016.param

## Go through the models, plot the original spectra and transformed spectra at $t=1$ s.

### The GCD model

In [ ]:
sn1 = Wright_2016(mechanism='GCD')
fig = plot_spectra(sn1, time=1*u.s)

### The DDT model

In [ ]:
sn2 = Wright_2016(mechanism='DDT')
fig = plot_spectra(sn2, time=1*u.s)

### Compute the total number of events in a JUNO like detector for a PISN at 10 kpc

In [ ]:
energies = np.linspace(0,10,1001)<<u.MeV
distance = 1 * u.kpc

transformation = AdiabaticMSW(MixingParameters())

rc = RateCalculator()

times    = sn1.get_time()
fluence = sn1.get_flux(t=times, E=energies, distance=distance, flavor_xform=transformation).integrate_or_sum('time')
dNdE = aggregate( rc.run(fluence, detector='scint20kt', detector_effects=False) )
Nevents = dNdE["total"].integrate_or_sum('energy').array.squeeze() 
print("GCD", Nevents)

times    = sn2.get_time()
fluence = sn2.get_flux(t=times, E=energies, distance=distance, flavor_xform=transformation).integrate_or_sum('time')
dNdE = aggregate( rc.run(fluence, detector='scint20kt', detector_effects=False) )
Nevents = dNdE["total"].integrate_or_sum('energy').array.squeeze() 
print("DDT",  Nevents)